# 04 — Matrice de couverture cross-notebooks

**EPIC #10355** — FallacyDetection, Phase 1 / synthèse. Ce notebook conclut la
Phase 1 en rendant **mesurable** la couverture de la série : chaque notebook de
la série vient de la même veine (axée « matériau de Phase 2 »), mais aucun ne
disait *ce qu'il couvre*. La matrice N×M ci-dessous le dit, en lisant les
notebooks eux-mêmes (aucune déclaration à la main).

**Axes mesurés** : colonnes M = **sophismes** (types exposés par les inventaires
du notebook), **formalismes** (cellules qui posent la paire formel/informel),
**domaines** (sources/datasets externes nommés), et la colonne **preuve**
(exécution réelle, erreurs, figures).

La lecture des notebooks est déléguée à l'organe canonique
`scripts/notebook_tools/fallacy_coverage_matrix.py` : cette cellule et les
suivantes **consomment** le module, elles ne réimplémentent pas sa mesure
(leçon #13921 — organe importable + consommateur externe).


In [1]:
import sys
from pathlib import Path

# Robuste au cwd : papermill peut lancer depuis repo-root ou le dossier notebook.
_CANDIDATES = [
    Path("scripts/notebook_tools/fallacy_coverage_matrix.py"),
    Path("../scripts/notebook_tools/fallacy_coverage_matrix.py"),
    Path("../../scripts/notebook_tools/fallacy_coverage_matrix.py"),
    Path("../../../scripts/notebook_tools/fallacy_coverage_matrix.py"),
]
_ORGAN = next((c for c in _CANDIDATES if c.is_file()), None)
assert _ORGAN is not None, "Organe introuvable depuis cwd=" + str(Path.cwd())
sys.path.insert(0, str(_ORGAN.parent))

import fallacy_coverage_matrix as fcm

print("Organe charge :", _ORGAN.name)


Organe charge : fallacy_coverage_matrix.py


## 1. La matrice N×M — lecture directe des notebooks

La matrice est **calculée**, pas écrite à la main : chaque ligne reflète ce que
le notebook définit réellement (inventaires `LOGIC_13`/`mafalda_labels`/…,
paires formel/informel, sources externes nommées, cellules exécutées).


In [2]:
from pathlib import Path

# Le dossier de la série, robuste au cwd.
_series = [
    Path("MyIA.AI.Notebooks/GenAI/FallacyDetection"),
    Path("../GenAI/FallacyDetection"),
    Path("../../GenAI/FallacyDetection"),
]
SERIES_DIR = next((c for c in _series if c.is_dir()), None)
assert SERIES_DIR is not None, "Serie introuvable depuis cwd=" + str(Path.cwd())

rows = fcm.build_matrix(SERIES_DIR)
print(fcm.markdown_table(rows))


| Notebook | Cellules | Exécutées | Exercices | Sophismes | Formalismes | Domaines | Preuve |
|---|---:|---:|---:|---:|---:|---:|---|
| 01_taxonomy_intro | 6 | 6 (100%) | 3 | 5 | 2 | 0 | 100% exécutées, 0 erreur(s) |
| 02_fallacy_datasets_landscape | 12 | 12 (100%) | 3 | 0 | 0 | 3 | 100% exécutées, 0 erreur(s) |
| 03_taxonomy_coverage_gap | 7 | 7 (100%) | 3 | 36 | 0 | 0 | 100% exécutées, 0 erreur(s) |
| 04_coverage_matrix | 6 | 6 (100%) | 3 | 0 | 0 | 0 | 100% exécutées, 0 erreur(s) |
| **Série** | 31 | 31 | 12 | 41 | 2 | 3 | — |


## 2. La lecture visuelle — heatmap

Une heatmap rend la **forme** de la couverture lisible en un coup d'œil, là où
le tableau rend les valeurs exactes. Les axes sont normalisés par ligne-maximum
(pour chaque axe, le notebook qui couvre le plus vaut 1) — cela montre
*qui porte quoi*, pas des valeurs absolues.


In [3]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

labels, grid = fcm.heatmap_payload(rows)
mat = np.array(grid)

# Normalisation par ligne (axe -> 1 pour le maximum).
norm = np.zeros_like(mat, dtype=float)
for i in range(mat.shape[0]):
    mx = mat[i].max()
    norm[i] = mat[i] / mx if mx > 0 else 0

fig, ax = plt.subplots(figsize=(9, 4.2))
im = ax.imshow(norm, cmap="YlGnBu", aspect="auto")
ax.set_xticks(range(len(rows)))
ax.set_xticklabels([r["notebook"].replace(".ipynb", "") for r in rows], rotation=25, ha="right", fontsize=8)
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels, fontsize=9)
for i in range(len(labels)):
    for j in range(len(rows)):
        ax.text(j, i, f"{mat[i][j]:g}", ha="center", va="center", fontsize=8, color="black" if norm[i][j] < 0.75 else "white")
fig.colorbar(im, ax=ax, label="couverture normalisée (max=1)")
ax.set_title("Couverture de la série FallacyDetection — matrice N×M normalisée")
plt.tight_layout()
plt.savefig("assets/fallacy_coverage_matrix.png", dpi=120)
print("Heatmap sauvee dans assets/fallacy_coverage_matrix.png")


Heatmap sauvee dans assets/fallacy_coverage_matrix.png


La heatmap normalisée — chaque ligne est un axe, l'intensité dit quel
notebook porte la couverture :

![Matrice de couverture de la série FallacyDetection](assets/fallacy_coverage_matrix.png)


## 3. Exercices

La matrice révèle des trous de couverture. Les trois exercices ci-dessous
instrumentent ces trous — ils se complètent avec les données déjà en mémoire
(`rows`, `labels`, `norm`).


### Exercice 1 — Lire la ligne « domaines »

**Contexte.** Exécute (1) lit la matrice : quel notebook couvre le plus de
sources externes ? Pourquoi les deux autres en couvrent-elles zéro — est-ce un
défaut de contenu ou une différence de métier ?


In [4]:
# Exercice 1 — Ligne (domaines) : lire et interpréter
# Etape 1 : indexer la ligne domaines dans la matrice (rows[i][domaines]).
# Etape 2 : nommer le notebook qui porte la ligne et justifier l'ecart.
# Etape 3 : dire si l'ecart est un defaut de couverture ou un effet de perimetre.
resultat_ex1 = None  # TODO etudiant


### Exercice 2 — Les sophismes sont-ils bien répartis ?

**Contexte.** La colonne « sophismes » additionne les inventaires de chaque
notebook — mais un inventaire *n'est pas* une couverture : nommer un type dans
une liste n'est pas l'étiqueter sur un corpus. Écrire le test de la technique
dont le 03 a besoin pour que sa colonne soit honnête.


In [5]:
# Exercice 2 — Sophismes (N' vs N) : un inventaire n'est pas une couverture
# Etape 1 : ecrire une fonction couverture_effective(inventaire, corpus_etiquetes)
#          qui rend le ratio de types effectivement presents dans le corpus.
# Etape 2 : la verifier sur l'inventaire LOGIC_13 face a un corpus jouet.
# Indice : regarder cote couverture des feuilles dans 03 (academic vs feuilles).
def couverture_effective(inventaire, corpus_etiquetes):
    resultat_ex2 = None  # TODO etudiant
    return resultat_ex2


### Exercice 3 — Que manque-t-il à la matrice ?

**Contexte.** La matrice mesure quatre axes (sophismes, formalismes, domaines,
preuve). Un axe manque : le **niveau de difficulté** / le **type de tâche**
(étiqueter, choisir, argumenter). Proposer un cinquième axe et le rendre
mesurable à partir des cellules code du notebook — mêmes contraintes que les
autres axes (mécanique, reproductible, zéro déclaration à la main).


In [6]:
# Exercice 3 — Proposer un 5e axe mesurable
# Etape 1 : definir le nom du nouvel axe et sa regle de mesure.
# Etape 2 : implementer une fonction axe_nouveau(notebook_path) -> int
#          qui compte les cellules code de type tache donnee.
# Etape 3 : la verifier sur le notebook 01 (3 exercices attendus).
def axe_nouveau(notebook_path):
    resultat_ex3 = None  # TODO etudiant
    return resultat_ex3


## 4. Conclusion — usage de la matrice Phase 2

La matrice est le **chantier de la Phase 2** : elle nomme où la série est déjà
forte (inventaires du 03, sources du 02) et où elle est vide (formalismes,
domaines dans les 01 et 03). Un Future dataset builder relira cette matrice comme
**carte de départ** — pas comme un verdict.